**TÍTULO**

F3N4 — Triagem de variáveis por correlação defasada e redução de dimensionalidade (PCA/EOF)

**CONTEXTO**

O sistema ENSO é altamente colinear: D20, OHC, WWV, SSH e temperaturas de subsuperfície compartilham a memória de recarga (Meinen & McPhaden, 2000), e os campos atmosféricos respondem coletivamente ao mesmo gradiente de SST. Um baseline estatístico linear rigoroso exige triagem por significância e compressão da redundância antes de qualquer modelagem.

**DESAFIO**

Hipótese: um subconjunto de variáveis lidera a SSTA de Niño 3.4 em lags de semanas a meses, e poucos componentes principais concentram ≥ 80% da variância do bloco preditor. Objetivos: (i) correlação cruzada defasada preditor(t−lag) × SSTA(t); (ii) descartar variáveis sem significância (p < 0,05) em nenhuma janela; (iii) PCA no bloco filtrado retendo ≥ 80% da variância; (iv) interpretar sinais dos coeficientes.

**METODOLOGIA**

Anomalias normalizadas da F3N1 (sazonalidade removida de TODAS as variáveis antes da modelagem). Lags de 1 a 52 semanas. Pearson e Spearman com N efetivo de Bretherton et al. (1999) para autocorrelação serial; teste t bicaudal com graus de liberdade N_eff − 2. Triagem: mantém-se a variável se p < 0,05 em pelo menos um lag (em qualquer dos dois coeficientes). PCA por SVD (pacote eofs quando disponível) na matriz filtrada; retêm-se os primeiros componentes até acumular 80% da variância (Wilks, 2011).

**RESULTADOS ESPERADOS**

TabF3N4_correlacoes_defasadas.csv, TabF3N4_triagem.csv (mantidas/descartadas), TabF3N4_variancia_pca.csv, TabF3N4_loadings.csv; FigF3N4_mapa_lag_correlacao (heatmap r × lag), FigF3N4_scree e FigF3N4_loadings_pc1_pc2. Coeficientes positivos: forçantes diretamente proporcionais à SSTA; negativos: forçantes que freiam ou invertem o fenômeno.

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. MEINEN, C. S.; McPHADEN, M. J. Observations of Warm Water Volume Changes in the Equatorial Pacific and Their Relationship to El Niño and La Niña. Journal of Climate, v. 13, p. 3551-3559, 2000. DOI: https://doi.org/10.1175/1520-0442(2000)013<3551:OOWWVC>2.0.CO;2
2. WILKS, D. S. Statistical Methods in the Atmospheric Sciences. 3. ed. Academic Press, 2011. DOI: https://doi.org/10.1016/C2010-0-66249-4
3. BRETHERTON, C. S. et al. The Effective Number of Spatial Degrees of Freedom of a Time-Varying Field. Journal of Climate, v. 12, p. 1990-2009, 1999.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
tabela_z = f3.table_dir() / 'TabF3N1_zscores_semanais.csv'
if tabela_z.exists():
    zscores = pd.read_csv(tabela_z, index_col=0, parse_dates=True)
else:
    zscores = f3.deseason_and_zscore(master[fisicas])
alvo = zscores['nino34_ssta']
preditores = zscores.drop(columns=['nino34_ssta'])
lags = [1, 2, 3, 4, 6, 8, 13, 17, 22, 26, 33, 39, 45, 52]
pearson = f3.lagged_correlations(preditores, alvo, lags_weeks=lags, method='pearson')
spearman = f3.lagged_correlations(preditores, alvo, lags_weeks=lags, method='spearman')
correlacoes = pearson.merge(spearman, on=['variavel', 'lag_semanas'], suffixes=('_pearson', '_spearman'))
f3.save_table(correlacoes, 'TabF3N4_correlacoes_defasadas', index=False)
correlacoes.head()

[tabela] TabF3N4_correlacoes_defasadas.csv (602x10) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


,variavel,lag_semanas,r_pearson,n_pearson,n_eff_pearson,p_valor_pearson,r_spearman,n_spearman,n_eff_spearman,p_valor_spearman
0,d20_m,1,0.517383,2339,49.0,0.000141,0.534972,2339,49.0,0.000075
1,d20_m,2,0.543182,2338,48.9,0.000056,0.556718,2338,48.9,0.000033
2,d20_m,3,0.565158,2337,48.9,0.000024,0.574847,2337,48.9,0.000016
3,d20_m,4,0.584616,2336,48.9,0.000011,0.591133,2336,48.9,0.000008
4,d20_m,6,0.618727,2334,48.9,0.000002,0.620272,2334,48.9,0.000002


In [3]:
significativa = correlacoes.assign(
    sig=lambda d: (d['p_valor_pearson'] < 0.05) | (d['p_valor_spearman'] < 0.05)
).groupby('variavel')['sig'].any()
mantidas = sorted(significativa[significativa].index)
descartadas = sorted(significativa[~significativa].index)
triagem = pd.DataFrame({'variavel': list(significativa.index),
                        'mantida_p_menor_005': significativa.values})
f3.save_table(triagem.sort_values('variavel'), 'TabF3N4_triagem', index=False)
print(f'mantidas: {len(mantidas)} | descartadas: {len(descartadas)}')
print('descartadas:', descartadas if descartadas else 'nenhuma')

[tabela] TabF3N4_triagem.csv (43x2) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
mantidas: 42 | descartadas: 1
descartadas: ['t700m']


In [4]:
mapa = correlacoes.pivot(index='variavel', columns='lag_semanas', values='r_pearson').loc[mantidas]
f3.save_table(mapa, 'FigF3N4_mapa_lag_correlacao_dados')
fig, ax = plt.subplots(figsize=(10, max(6, 0.28 * len(mapa))))
im = ax.imshow(mapa.to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(mapa.columns)), mapa.columns)
ax.set_yticks(range(len(mapa.index)), mapa.index, fontsize=7)
ax.set_xlabel('Lag (semanas): preditor(t-lag) x SSTA(t)')
ax.set_title('Correlacao defasada com SSTA Nino 3.4 (variaveis significativas)')
fig.colorbar(im, ax=ax, label='r de Pearson')
fig.tight_layout()
f3.save_figure(fig, 'FigF3N4_mapa_lag_correlacao')
plt.close(fig)

[tabela] FigF3N4_mapa_lag_correlacao_dados.csv (42x14) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N4_mapa_lag_correlacao.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3


In [5]:
scores, loadings, variancia, n_pcs = f3.pca_from_zscores(preditores[mantidas], variance_target=0.80)
f3.save_table(variancia.to_frame(), 'TabF3N4_variancia_pca')
f3.save_table(loadings, 'TabF3N4_loadings')
f3.save_table(scores, 'TabF3N4_scores_pca')
print(f'{n_pcs} componentes retem {100 * variancia.sum():.1f}% da variancia')

fig, ax = plt.subplots(figsize=(7, 4))
acumulada = variancia.cumsum()
ax.bar(range(1, n_pcs + 1), 100 * variancia.values, label='individual')
ax.plot(range(1, n_pcs + 1), 100 * acumulada.values, 'ko-', label='acumulada')
ax.axhline(80, color='r', ls='--', lw=0.8, label='alvo 80%')
ax.set_xlabel('Componente principal')
ax.set_ylabel('Variancia explicada (%)')
ax.set_title('Scree plot do bloco preditor filtrado')
ax.legend()
fig.tight_layout()
f3.save_figure(fig, 'FigF3N4_scree')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axhline(0, color='0.7', lw=0.6); ax.axvline(0, color='0.7', lw=0.6)
ax.scatter(loadings['PC1'], loadings['PC2'], s=12)
for nome, linha in loadings.iterrows():
    ax.annotate(nome, (linha['PC1'], linha['PC2']), fontsize=6)
ax.set_xlabel('Loading PC1'); ax.set_ylabel('Loading PC2')
ax.set_title('Loadings PC1 x PC2 — sinal positivo refor\u00e7a, negativo freia a SSTA')
fig.tight_layout()
f3.save_figure(fig, 'FigF3N4_loadings_pc1_pc2')
plt.close(fig)

orientacao = loadings['PC1'].sort_values()
print('Interpretacao PC1 — extremos negativos (freiam o fenomeno):')
print(orientacao.head(5).round(3).to_string())
print('Interpretacao PC1 — extremos positivos (reforcam o fenomeno):')
print(orientacao.tail(5).round(3).to_string())

[tabela] TabF3N4_variancia_pca.csv (3x1) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N4_loadings.csv (42x3) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N4_scores_pca.csv (2339x3) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
3 componentes retem 82.3% da variancia


[figura] FigF3N4_scree.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3


[figura] FigF3N4_loadings_pc1_pc2.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
Interpretacao PC1 — extremos negativos (freiam o fenomeno):
ssr_anom    -0.768
slhf_anom   -0.321
sshf_anom   -0.151
wwv         -0.034
t150m       -0.027
Interpretacao PC1 — extremos positivos (reforcam o fenomeno):
q500_anom    0.011
q850_anom    0.011
tcwv_anom    0.014
v200_anom    0.014
str_anom     0.526
